In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import numpy as np

R_CATHODE = 46.29e-3 / 2   # m  — cathode sphere radius
R_WALL    = 280e-3   / 2   # m  — chamber wall radius

radii = np.linspace(R_CATHODE, R_WALL, 400)  # 400 radial points from cathode to wall

class PlasmaMLP(nn.Module):
    '''
    input: radius, Power, Voltage, Current,Pressure , rat1, rat2, rat3, rat4, rat5  (10 features)
    output: ne, Te (2 features)
    '''
    def __init__(self, input_size=10, hidden_size=128, output_size=2):
        super(PlasmaMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, output_size)
        )

    def forward(self, x):
        return self.network(x)
    
def train_plasma_model_pt(model, X_train_tensor, y_train_tensor, epochs=100, batch_size=32):
    dataset = TensorDataset(X_train_tensor, y_train_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model


In [13]:
import pandas as pd
import numpy as np
import json

def json_to_dataframe(json_file_path):
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    rows = []
    
    for run_id, content in data.items():
        # Extract scalar metadata
        current = content['Current']
        voltage = content['Voltage']
        pressure = content['Pressure']
        ratios = content['Peak_Ratios']
        temps = content['Electron_Temperatures']
        dens = content['Electron_Densities']
        
        # Create an index/radius for the spatial profile
        # If your data is 100 points, create a range from 0 to 99 
        # (or define your actual radius values here)
        
        for i, (temp,dens) in enumerate(zip(temps,dens)):
            rows.append({
                'run_id': run_id,
                'radius': radii[i],
                'current': current,
                'voltage': voltage,
                'power': voltage * current,  # Calculate power
                'pressure': pressure,
                # Expand ratios into separate columns
                'rat1': ratios[0] if len(ratios) > 0 else np.nan,
                'rat2': ratios[1] if len(ratios) > 1 else np.nan,
                'rat3': ratios[2] if len(ratios) > 2 else np.nan,
                'rat4': ratios[3] if len(ratios) > 3 else np.nan,
                'rat5': ratios[4] if len(ratios) > 4 else np.nan,
                'te': temp,
                'ne':dens
            })
            
    return pd.DataFrame(rows)

# Load and process
df_plasma = json_to_dataframe('training_data.json')

# Quick clean-up: Remove NaNs if necessary
df_plasma = df_plasma.dropna()

print(df_plasma.head())
print(len(df_plasma))

  run_id    radius  current  voltage   power  pressure      rat1      rat2  \
0      0  0.023145     60.0     38.0  2280.0      51.5  1.504546  2.256411   
1      0  0.023438     60.0     38.0  2280.0      51.5  1.504546  2.256411   
2      0  0.023731     60.0     38.0  2280.0      51.5  1.504546  2.256411   
3      0  0.024024     60.0     38.0  2280.0      51.5  1.504546  2.256411   
4      0  0.024316     60.0     38.0  2280.0      51.5  1.504546  2.256411   

       rat3      rat4      rat5        te            ne  
0  1.063636  1.646767  0.396006  1.583085  1.576645e+17  
1  1.063636  1.646767  0.396006  1.516075  1.488242e+17  
2  1.063636  1.646767  0.396006  1.412786  1.405609e+17  
3  1.063636  1.646767  0.396006  1.354909  1.329036e+17  
4  1.063636  1.646767  0.396006  1.342445  1.257891e+17  
17600


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def evaluate_plasma_model(df, features, targets, model_class):
    X = df[features].values
    
    y_ne = df[targets[0]].values.reshape(-1, 1)
    y_te = df[targets[1]].values.reshape(-1, 1)
    y = np.hstack([y_ne, y_te])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. Scale
    scaler_x = StandardScaler().fit(X_train)
    scaler_y = StandardScaler().fit(y_train)

    X_train_t = torch.tensor(scaler_x.transform(X_train), dtype=torch.float32)
    y_train_t = torch.tensor(scaler_y.transform(y_train), dtype=torch.float32)
    
    X_test_t = torch.tensor(scaler_x.transform(X_test), dtype=torch.float32)
    y_test_t = torch.tensor(scaler_y.transform(y_test), dtype=torch.float32)

    # 3. Initialize and Train
    model = model_class() # Instantiate the PlasmaMLP class
    model = train_plasma_model_pt(model, X_train_t, y_train_t)

    # Evaluate
    model.eval() 
    with torch.no_grad():
        preds = model(X_test_t)
        
        # Split the tensors by column (0 for ne, 1 for te)
        pred_ne = preds[:, 0]
        true_ne = y_test_t[:, 0]
        
        pred_te = preds[:, 1]
        true_te = y_test_t[:, 1]

        # Calculate individual MSEs
        mse_ne = nn.functional.mse_loss(pred_ne, true_ne).item()
        mse_te = nn.functional.mse_loss(pred_te, true_te).item()
        
        # Total MSE (for context)
        mse_total = nn.functional.mse_loss(preds, y_test_t).item()
    
    print(f"Total MSE: {mse_total:.4f}")
    print(f"Density (ne) MSE: {mse_ne:.4f}")
    print(f"Temperature (te) MSE: {mse_te:.4f}")
    
    return mse_ne, mse_te, model, scaler_x, scaler_y



In [ ]:
# --- Execution ---
features = ['radius', 'power', 'voltage', 'current', 'pressure', 'rat1', 'rat2', 'rat3', 'rat4', 'rat5']
targets = ['ne', 'te']

mse_ne,mse_te, trained_model, scaler_x, scaler_y = evaluate_plasma_model(df_plasma, features, targets, PlasmaMLP)


Total MSE: 0.0161
Density (ne) MSE: 0.0213
Temperature (te) MSE: 0.0108


ValueError: too many values to unpack (expected 4)